In [ ]:
print('===== CPU BACKEND BITEXACT V2 - acc=0 then add gdw at end =====')
# 1/4 - clone both: main (CPU native C) + cuda branch (cpu backend + linear_dw_cpu)
import subprocess, os
print('===== CPU BACKEND BITEXACT V1 - CORRECT FILE =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build CPU native .so (tkv_linear_dw.c) + .tkv-generated CPU C .so
import subprocess, shutil, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('CPU native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:3000])
print('native so exists',os.path.exists(root+'/tkv_simd_native.so'))
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/linear_dw_cpu.cpu.c'; so=ex+'/linear_dw_cpu_cpu.so'
print('generated C exists',os.path.exists(cc))
print('has fmaf =', 'fmaf(' in open(cc).read())
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv-generated C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:3000])
print('tkv so exists',os.path.exists(so))

In [ ]:
# 3/4 - bit-exact compare CPU native vs .tkv-generated C
import ctypes, numpy as np
root='/content/TVmain/Release/tools/simd_native'
cpu=ctypes.CDLL(root+'/tkv_simd_native.so')
cpu.LinearDwFused.argtypes=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,
                            ctypes.POINTER(ctypes.c_float),ctypes.c_int,
                            ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_int,
                            ctypes.c_int,ctypes.c_int,ctypes.c_int]
cpu.LinearDwFused.restype=ctypes.c_int
F32=ctypes.c_float
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/linear_dw_cpu_cpu.so')
tkv.tkv_cpu_linear_dw_cpu.argtypes=[ctypes.POINTER(F32)]*3+[ctypes.c_int]*3
tkv.tkv_cpu_linear_dw_cpu.restype=ctypes.c_int
rng=np.random.default_rng(7)
for n,in_d,out_d in [(8,16,32),(8,64,24),(8,8,64),(4,32,48)]:
    x=rng.standard_normal(n*in_d).astype(np.float32)
    gy=rng.standard_normal(n*out_d).astype(np.float32)
    g0=rng.standard_normal(in_d*out_d).astype(np.float32)
    a=g0.copy(); b=g0.copy()
    rc=cpu.LinearDwFused(x.ctypes.data_as(ctypes.POINTER(F32)),x.size,
                         gy.ctypes.data_as(ctypes.POINTER(F32)),gy.size,
                         a.ctypes.data_as(ctypes.POINTER(F32)),a.size,0,n,in_d,out_d)
    rc2=tkv.tkv_cpu_linear_dw_cpu(x.ctypes.data_as(ctypes.POINTER(F32)),
                                  gy.ctypes.data_as(ctypes.POINTER(F32)),
                                  b.ctypes.data_as(ctypes.POINTER(F32)),n,in_d,out_d)
    diff=np.count_nonzero(a.view(np.uint32)!=b.view(np.uint32))
    print('n=%d in=%d out=%d rc=%d/%d bitwise=%s diff=%d/%d maxabs=%.3g'%(
          n,in_d,out_d,rc,rc2,diff==0,diff,a.size,np.abs(a-b).max()))

In [ ]:
# 4/4 - summary
print('If ALL bitwise=True -> STEP5 PASS: continue to STEP6 (replace tkv_linear_dw.c).')
print('If any diff -> STOP, keep native C (plan b-step-5).')